In [ ]:

import os
import numpy as np
from matplotlib import pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
from matplotlib import colors
from matplotlib.path import Path


# output_file_path = '/share/nas/kelvinw/meti/'
# output_file = os.path.join(output_file_path,'output_m15.dat')

filename = '/share/nas/kelvinw/meti/output_m15.dat'

def drop_null_teff_lum(filename):
    """
    Takes the output file from PYSSED, makes a df and filters it to remove null
    effective temperatures and luminosities
    """

    df = pd.read_csv(filename,delimiter='\t',comment='#',header=None)
    
    ids = df.iloc[:,0].values.astype('int')
    ra = df.iloc[:,1].values.astype('float')
    dec = df.iloc[:,2].values.astype('float')
    eff_temp = df.iloc[:,9].values.astype('float')
    lum = df.iloc[:,10].values.astype('float')


    selected_columns = {
        'ID': ids,
        'RA': ra,
        'Dec': dec,
        'Teff': eff_temp,
        'Luminosity': lum,
    }


    df_hst = pd.DataFrame(selected_columns)

    ## Write an output file to confirm everything has worked
    file_selected_columns = 'df_selected_columns.dat'
    os.system(f"rm -r {file_selected_columns}")
    file_selected_columns_path = os.path.join(os.path.dirname(filename),file_selected_columns)
    df_hst.to_csv(file_selected_columns_path, index=False)
    
    # # Filter rows where either Teff or Luminosity is zero
    zero_values_df = df_hst[(df_hst['Teff'] == 0.0) | (df_hst['Luminosity'] == 0.0)]

    # Get the IDs of these rows
    ids_with_zero_values = zero_values_df['ID'].tolist()
    # print(len(ids_with_zero_values))

    rows_to_drop = df_hst[df_hst['ID'].isin(ids_with_zero_values)].index ## approach agrees with the deleting -- same indices returned
    print(f"Dropping rows: {rows_to_drop}")
    df_hst = df_hst.drop(rows_to_drop)

    ## Write an output file to confirm everything has worked
    file_filtered = 'df_filtered.dat'
    os.system(f"rm -r {file_filtered}")
    file_filtered_path = os.path.join(os.path.dirname(filename),file_filtered)
    df_hst.to_csv(file_filtered_path, index=False)

    return df_hst

    


In [ ]:

def select_stars(vertex, df_hst):
    """
    Accepts a dictionary of vertices and selects stars inside the polygons defined by these vertices.
    
    Args:
    - vertex_dict: Dictionary where keys are descriptive names for the vertex sets and values are lists of vertices.
    - df_hst: DataFrame containing star data with columns 'Teff', 'Luminosity', 'ID', 'RA', 'Dec'.
    
    Returns:
    - filtered_data_dict: Dictionary where keys are the same as input vertex_dict and values are DataFrames of stars inside the polygons.
    """
    eff_temp_hst = df_hst['Teff'].values.astype('float')
    lum_hst = df_hst['Luminosity'].values.astype('float')
    
    def transform_to_log_scale(vertices):
        return [(x, np.log10(y)) for x, y in vertices]

    filtered_data_dict = {}

    for name, vertices in vertex_dict.items():
        print(name,vertices)
        # Transform vertices for logarithmic scale
        vertices_log = transform_to_log_scale(vertices)
        polygon_log = Path(vertices_log)

        # Transform star data to logarithmic scale
        points_log = np.vstack((eff_temp_hst, np.log10(lum_hst))).T
        inside_polygon = polygon_log.contains_points(points_log)

        # Filter stars inside the polygon
        filtered_data = df_hst.loc[inside_polygon, ['ID', 'Luminosity', 'Teff', 'RA', 'Dec']]
        filtered_data_dict[name] = filtered_data

    return filtered_data_dict

def plot_hrd(vertex_dict):
    
    df_hst = drop_null_teff_lum(filename)

    eff_temp_hst = df_hst['Teff'].values.astype('float')
    lum_hst = df_hst['Luminosity'].values.astype('float')

    # Plotting
    fig, ax = plt.subplots(figsize=(16, 12))

    # Plot all stars
    ax.scatter(eff_temp_hst, lum_hst, s=3, color='blue', label='All Stars')

    colors = ['red', 'black', 'green', 'orange', 'purple']
    filtered_data_dict = select_stars(vertex_dict, df_hst)
    
    for i, (name, filtered_data) in enumerate(filtered_data_dict.items()):
        filtered_temp = filtered_data['Teff'].values.astype('float')
        filtered_lum = filtered_data['Luminosity'].values.astype('float')

        # Plot stars inside the polygon
        color = colors[i % len(colors)]  # Use modulus to cycle colors if more regions than colors
        ax.scatter(filtered_temp, filtered_lum, s=3, color=color, label=name)

        print(f"Region '{name}' plotted with color {color}.")
        print(f"Number of stars in {name}: {len(filtered_data)}")


    # Logarithmic scale for y-axis
    ax.set_yscale('log')

    # Setting x and y limits
    ax.set_xlim(0, 20000)
    ax.set_ylim(10e-4, 10e4)

    # Labels and title
    ax.set_xlabel('Temperature (K)', fontsize=14)
    ax.set_ylabel('Luminosity', fontsize=14)
    ax.set_title('Hertzsprung-Russell Diagram')

    # Inverting the x-axis (temperature decreases to the right)
    ax.invert_xaxis()

    # Custom y-axis ticks
    ax.set_yticks([0.01, 0.1, 1, 10, 100, 1000, 10000])
    ax.get_yaxis().set_major_formatter(plt.ScalarFormatter())

    # Custom x-axis ticks
    x_ticks = range(2000, 20001, 2000)
    x_tick_labels = [str(tick) for tick in x_ticks]
    ax.set_xticks(x_ticks)
    ax.set_xticklabels(x_tick_labels)

    # Legend
    ax.legend()

    plt.show()

    return filtered_data_dict


vertex_dict = {
    'Horizontal Branch': [
        (5700, 20), 
        (40000, 20), 
        (40000, 100), 
        (5700, 100)
    ],
    'White Dwarfs': [
        (5200, 0.005), 
        (5300, 0.06), 
        (10000, 1),
        (14000, 1),
    ]
}

filtered_data_dict = plot_hrd(vertex_dict)

# filtered_data will now contain the filtered DataFrames for each set of vertices
for region, data in filtered_data_dict.items():
    print(f"{region} data:")
    print(data.head())




In [ ]:
def plot_density_hrd():

    """
    Uses the filtered data frame to plot the HRD 
    
    """
    df_hst = drop_null_teff_lum(filename)

    
    eff_temp_hst = df_hst.iloc[:,3].values.astype('float')
    lum_hst = df_hst.iloc[:,4].values.astype('float')


    # Log transform the data to handle wide range of values
    log_eff_temp_hst = np.log10(eff_temp_hst)
    log_lum_hst = np.log10(lum_hst)
    
    # Create hexbin plot
    fig, ax = plt.subplots(figsize=(16, 12))
    
    # Plotting the hexbin with logarithmic data
    hb = ax.hexbin(log_eff_temp_hst, log_lum_hst, gridsize=2000, cmap='inferno', bins='log', mincnt=1)
    
    # Adding colorbar
    cb = fig.colorbar(hb, ax=ax)
    cb.set_label('log (Number of Stars)', fontsize=14)
    cb.ax.tick_params(labelsize=14)  # Set the font size of the color bar tick labels
    
    
    
    ax.set_xscale('linear')  # Log-transformed data, so use linear scale
    ax.set_yscale('linear')  # Log-transformed data, so use linear scale
    ax.set_xlabel('Log10 Effective Temperature (K)', fontsize=14)
    ax.set_ylabel('Log10 Luminosity', fontsize=14)
    
    ax.set_xlim(3.47,4.4)
    ax.set_ylim(-2.5,3)
    
    
    ax.tick_params(axis='x',labelsize=14)
    ax.tick_params(axis='y',labelsize=14)
    plt.gca().invert_xaxis()
    
    
    # plt.savefig('hrd_hst.pdf',dpi=300)
    
    plt.show()


# plot_density_hrd()